# Bradford Bulls — Colab Setup & Run

This notebook:
1. Clones the project repo
2. Installs all dependencies
3. Downloads model weights (Real-ESRGAN)
4. Mounts Google Drive (for video/metadata access)
5. Runs the frame reconstruction pipeline

**Runtime**: GPU → T4 (Runtime → Change runtime type → T4 GPU)

## 1. Clone Project Repo

In [ ]:
import os

REPO_DIR = '/content/bradford_bulls'

if os.path.exists(REPO_DIR):
    print(f'Repo already exists at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/hoamx2602/bradford_bulls.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 2. Install Dependencies

After this cell finishes, **Restart Runtime** if prompted (Colab may require restart after torch/torchvision upgrades).

In [ ]:
# Core dependencies from requirements.txt
!pip install -q opencv-python-headless pandas tqdm matplotlib

# Install gdown just in case
!pip install -q gdown


## 4. Download Model Weights

In [ ]:
WEIGHTS_DIR = '/content/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

REALESRGAN_WEIGHTS = f'{WEIGHTS_DIR}/RealESRGAN_x2plus.pth'

# Real-ESRGAN weights (~64 MB)
if not os.path.exists(REALESRGAN_WEIGHTS):
    print('Downloading Real-ESRGAN weights...')
    !wget -q --show-progress -O {REALESRGAN_WEIGHTS} \
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth'
else:
    print(f'Real-ESRGAN weights already exist: {REALESRGAN_WEIGHTS}')

# Verify
for w in [REALESRGAN_WEIGHTS]:
    size_mb = os.path.getsize(w) / 1e6
    print(f'  ✓ {os.path.basename(w)} ({size_mb:.0f} MB)')


## 5. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 6. Verify GPU & Environment

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {mem:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

print(f'\nWorking directory: {os.getcwd()}')
print(f'Weights dir: {os.listdir(WEIGHTS_DIR)}')

## 7. Configure Paths

Edit the paths below to match your Google Drive structure.

In [ ]:
# ══════════════════════════════════════════════════════════
# EDIT THESE PATHS
# ══════════════════════════════════════════════════════════

VIDEO_PATH = '/content/drive/MyDrive/Bradford_Bulls/videos/M06_black_1080p.mp4'
CSV_PATH   = '/content/drive/MyDrive/Bradford_Bulls/metadata/M06_black_1080p_v3_index.csv'
OUTPUT_DIR = '/content/drive/MyDrive/Bradford_Bulls/frames_reconstructed/M06_black_1080p'

# ══════════════════════════════════════════════════════════

# Validate
for label, path in [('Video', VIDEO_PATH), ('CSV', CSV_PATH)]:
    exists = os.path.exists(path)
    status = 'OK' if exists else 'NOT FOUND'
    print(f'{label}: {path} [{status}]')

## 8. Run Reconstruction — Test Mode (10 frames)

Start with test mode to verify everything works and check visual quality.

In [ ]:
os.chdir(REPO_DIR)
!python run_reconstruction.py \
    --video "{VIDEO_PATH}" \
    --csv "{CSV_PATH}" \
    --output "{OUTPUT_DIR}" \
    --weights-dir {WEIGHTS_DIR} \
    --test

## 9. Preview Results

Visual comparison of original vs reconstructed frames.

In [ ]:
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Load reconstruction metadata
match_id = Path(VIDEO_PATH).stem
recon_csv = Path(OUTPUT_DIR) / f'{match_id}_enhancement_index.csv'
results_df = pd.read_csv(recon_csv)

print(f'Reconstructed {len(results_df)} frames')
print(f'Average improvement: {results_df["improvement"].mean():.2f}x')
print()
print(results_df[['filename', 'method', 'orig_sharpness', 'recon_sharpness', 'improvement']])

In [ ]:
# Side-by-side comparison: top improved frames
n_preview = min(6, len(results_df))
preview_df = results_df.sort_values('improvement', ascending=False).head(n_preview)

video_cap = cv2.VideoCapture(VIDEO_PATH)

fig, axes = plt.subplots(n_preview, 2, figsize=(20, 5 * n_preview))
if n_preview == 1:
    axes = axes.reshape(1, -1)

for i, (_, row) in enumerate(preview_df.iterrows()):
    # Original
    video_cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, original = video_cap.read()
    if ret:
        axes[i, 0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'Original — sharpness: {row["orig_sharpness"]:.0f}', fontsize=12)
    axes[i, 0].axis('off')

    # Reconstructed
    recon_path = Path(OUTPUT_DIR) / row['filename']
    if recon_path.exists():
        recon = cv2.imread(str(recon_path))
        axes[i, 1].imshow(cv2.cvtColor(recon, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(
        f'Reconstructed ({row["method"]}) — sharpness: {row["recon_sharpness"]:.0f} '
        f'({row["improvement"]:.2f}x)', fontsize=12
    )
    axes[i, 1].axis('off')

video_cap.release()
plt.tight_layout()
plt.show()

## 10. Run Production (All Frames)

Only run this after verifying test results look good.

In [ ]:
os.chdir(REPO_DIR)
!python run_reconstruction.py \
    --video "{VIDEO_PATH}" \
    --csv "{CSV_PATH}" \
    --output "{OUTPUT_DIR}" \
    --weights-dir {WEIGHTS_DIR}